<a href="https://colab.research.google.com/github/anupbagale/python-for-ai-ml/blob/main/05-files-and-data/02_exercises.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 5: Exercises

Twelve exercises: four Easy, five Medium and three Hard. They're all about reading, writing, checking and cleaning data files.

The checks create real temporary files and folders for each test, run your code on them, then look at what was read or written. Nothing touches your own files.

**Run the setup cell first.**

Rules:

- Standard library only: `pathlib`, `csv`, `json`, `collections` and `re` are all fine.
- Always pass `encoding="utf-8"` (or `"utf-8-sig"` where the exercise needs it) when opening text files.
- Exercise 5 uses your `parse_value` from exercise 1, so do that one first.
- Solutions are in `03_solutions.ipynb`.

In [ ]:
# Setup: run this cell once before you start.
# It defines the check functions used below.

import copy
import math
import csv
import inspect
import json
import tempfile
from pathlib import Path


class Raises:
    """Marks a test case where the code is expected to raise an error."""
    def __init__(self, exc):
        self.exc = exc

    def __repr__(self):
        return f"{self.exc.__name__} to be raised"


def _same(a, b, tol=1e-6):
    if isinstance(a, float) or isinstance(b, float):
        if not isinstance(a, (int, float)) or not isinstance(b, (int, float)):
            return False
        if math.isnan(a) or math.isnan(b):
            return False
        return math.isclose(a, b, rel_tol=tol, abs_tol=tol)
    if isinstance(a, (list, tuple)) and isinstance(b, (list, tuple)):
        return len(a) == len(b) and all(_same(x, y) for x, y in zip(a, b))
    if isinstance(a, dict) and isinstance(b, dict):
        return a.keys() == b.keys() and all(_same(a[k], b[k]) for k in a)
    return a == b


def _short(x, limit=70):
    text = repr(x)
    return text if len(text) <= limit else text[:limit - 3] + "..."


def _scenarios(name, cases):
    for description, scenario, expected, hint in cases:
        try:
            result = scenario()
        except Exception as e:
            if isinstance(expected, Raises) and isinstance(e, expected.exc):
                continue
            print(f"FAIL  {name}: {description}")
            print(f"      raised {type(e).__name__}: {e}")
            if isinstance(expected, Raises):
                print(f"      expected {expected!r}")
            else:
                print(f"      expected: {_short(expected)}")
            if hint:
                print(f"      hint: {hint}")
            return
        if isinstance(expected, Raises):
            print(f"FAIL  {name}: {description}")
            print(f"      expected {expected!r}, but nothing was raised")
            if hint:
                print(f"      hint: {hint}")
            return
        if not _same(result, expected):
            print(f"FAIL  {name}: {description}")
            print(f"      expected: {_short(expected)}")
            print(f"      got:      {_short(result)}")
            if hint:
                print(f"      hint: {hint}")
            return
    print(f"PASS  all {len(cases)} tests passed for {name}")


def _tmp() -> Path:
    """A fresh, empty temporary folder for one test."""
    return Path(tempfile.mkdtemp())


def _write(path, text, encoding="utf-8"):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text, encoding=encoding)
    return path


# ---- checks ----

def check_parse_value(f):
    typed = lambda text: (f(text), type(f(text)).__name__)
    _scenarios("parse_value", [
        ("integer", lambda: typed("42"), (42, "int"), "try int() before float()"),
        ("float", lambda: typed("3.5"), (3.5, "float"), None),
        ("scientific notation", lambda: f("1e-3"), 0.001, None),
        ("negative", lambda: f("-7"), -7, None),
        ("NA with spaces", lambda: f("  NA "), None, "strip first, then compare in lowercase"),
        ("empty string", lambda: f(""), None, None),
        ("question mark", lambda: f("?"), None, None),
        ("the text 'nan'", lambda: f("nan"), None, "float('nan') works, so check for missing markers before converting"),
        ("TRUE", lambda: typed("TRUE"), (True, "bool"), None),
        ("false", lambda: typed("false"), (False, "bool"), None),
        ("text is stripped", lambda: f("  Adelie "), "Adelie", None),
        ("text that starts like a number", lambda: f("3 g"), "3 g", None),
    ])


def check_count_lines(f):
    def count(text):
        return f(_write(_tmp() / "f.txt", text))

    _scenarios("count_lines", [
        ("three lines with a final newline", lambda: count("a\nb\nc\n"), 3, None),
        ("last line without a newline", lambda: count("a\nb\nc"), 3, "don't count newline characters; count lines"),
        ("empty file", lambda: count(""), 0, None),
        ("blank lines count too", lambda: count("a\n\nb\n"), 3, None),
        ("accepts a string path", lambda: f(str(_write(_tmp() / "g.txt", "x\ny\n"))), 2, None),
    ])


def check_save_json(f):
    def content():
        path = _tmp() / "out" / "nested" / "result.json"
        f({"b": 1, "a": [1, 2]}, path)
        return path.read_text(encoding="utf-8")

    def returns_path():
        path = _tmp() / "x.json"
        result = f({}, path)
        return isinstance(result, Path) and result == path

    def unicode():
        path = _tmp() / "u.json"
        f({"species": "Adélie"}, path)
        return "Adélie" in path.read_text(encoding="utf-8")

    def round_trip():
        path = _tmp() / "r.json"
        data = {"model": "knn", "params": {"k": 5}, "scale": True, "notes": None}
        f(data, path)
        return json.loads(path.read_text(encoding="utf-8")) == data

    _scenarios("save_json", [
        ("creates missing folders", lambda: (content() is not None), True, "path.parent.mkdir(parents=True, exist_ok=True)"),
        ("exact formatting", content, '{\n  "b": 1,\n  "a": [\n    1,\n    2\n  ]\n}\n',
         "indent=2, keep the key order, and end the file with a newline"),
        ("returns the path as a Path", returns_path, True, None),
        ("non-English text is written as-is", unicode, True, "ensure_ascii=False"),
        ("round trip", round_trip, True, None),
    ])


def check_find_files(f):
    def setup():
        root = _tmp()
        for name in ["a.csv", "b.txt", "sub/c.csv", "sub/deep/d.CSV", "sub/e.json"]:
            _write(root / name, "x")
        (root / "folder.csv").mkdir()
        return root

    _scenarios("find_files", [
        ("finds files in all subfolders", lambda: f(setup(), ".csv"), ["a.csv", "sub/c.csv", "sub/deep/d.CSV"],
         "search recursively (rglob), compare suffixes in lowercase, skip folders, return sorted relative paths"),
        ("other suffix", lambda: f(setup(), ".json"), ["sub/e.json"], None),
        ("suffix given in capitals", lambda: f(setup(), ".TXT"), ["b.txt"], None),
        ("nothing found", lambda: f(setup(), ".png"), [], None),
        ("accepts a string path", lambda: f(str(setup()), ".txt"), ["b.txt"], None),
    ])


def check_read_csv_records(f):
    def basic():
        path = _write(_tmp() / "people.csv", 'name,age,city\n"Smith, John",42,Kathmandu\nAna,NA,Pokhara\n')
        return f(path)

    def bom():
        path = _write(_tmp() / "excel.csv", "id,score\n1,0.5\n", encoding="utf-8-sig")
        return f(path)

    def empty_file():
        return f(_write(_tmp() / "e.csv", "id,score\n"))

    _scenarios("read_csv_records", [
        ("quoted commas and missing values", basic,
         [{"name": "Smith, John", "age": 42, "city": "Kathmandu"}, {"name": "Ana", "age": None, "city": "Pokhara"}],
         "use csv.DictReader, and convert every value with parse_value"),
        ("a file saved by Excel (with a BOM)", bom, [{"id": 1, "score": 0.5}],
         "open with encoding='utf-8-sig' so the first column isn't called '\\ufeffid'"),
        ("header only", empty_file, [], None),
    ])


def check_write_csv(f):
    def read_back(path):
        with open(path, encoding="utf-8", newline="") as fh:
            return list(csv.reader(fh))

    def basic():
        path = _tmp() / "out.csv"
        f([{"name": "Smith, John", "age": 42}, {"name": "Ana", "city": "Pokhara"}], path)
        return read_back(path)

    def returns_count():
        return f([{"a": 1}, {"a": 2}, {"a": 3}], _tmp() / "n.csv")

    def nones():
        path = _tmp() / "none.csv"
        f([{"a": None, "b": 0}], path)
        return read_back(path)

    _scenarios("write_csv", [
        ("columns from all records, in first-seen order", basic,
         [["name", "age", "city"], ["Smith, John", "42", ""], ["Ana", "", "Pokhara"]],
         "collect the keys from every record. Missing keys and None become empty fields"),
        ("None becomes empty, 0 stays 0", nones, [["a", "b"], ["", "0"]], "check `is None`, not truthiness"),
        ("returns the number of rows written", returns_count, 3, None),
    ])


def check_missing_report(f):
    records = [
        {"species": "Adelie", "mass": 3700, "sex": None},
        {"species": "Gentoo", "mass": None, "sex": None},
        {"species": "Gentoo", "mass": 5000},
        {"species": None, "mass": 3500, "sex": "male", "notes": "tagged"},
    ]
    _scenarios("missing_report", [
        ("counts, fractions and order", lambda: f(records),
         [("notes", 3, 0.75), ("sex", 3, 0.75), ("mass", 1, 0.25), ("species", 1, 0.25)],
         "a key that's absent from a record counts as missing too. Sort by count (high to low), then by name"),
        ("nothing missing", lambda: f([{"a": 1}, {"a": 2}]), [("a", 0, 0.0)], None),
        ("no records", lambda: f([]), [], None),
    ])


def check_fill_missing(f):
    train = [{"x": 1.0}, {"x": None}, {"x": 3.0}, {"x": 10.0}]
    test = [{"x": None}, {"x": 5.0}]

    def no_mutation():
        tr = [dict(r) for r in train]
        te = [dict(r) for r in test]
        f(tr, te, "x", "median")
        return (tr, te) == (train, test)

    def mode_text():
        tr = [{"c": "a"}, {"c": "b"}, {"c": "b"}, {"c": None}]
        return f(tr, [{"c": None}], "c", "mode")

    _scenarios("fill_missing", [
        ("median from train only", lambda: f(train, test, "x", "median"),
         ([{"x": 1.0}, {"x": 3.0}, {"x": 3.0}, {"x": 10.0}], [{"x": 3.0}, {"x": 5.0}], 3.0),
         "use only the non-missing training values: [1, 3, 10] has median 3"),
        ("mean", lambda: f(train, test, "x", "mean")[2], 14 / 3, None),
        ("even number of values", lambda: f([{"x": 1}, {"x": 2}, {"x": 4}, {"x": 9}], [], "x", "median")[2], 3.0,
         "with an even count, the median is the mean of the two middle values"),
        ("mode works for text", mode_text, ([{"c": "a"}, {"c": "b"}, {"c": "b"}, {"c": "b"}], [{"c": "b"}], "b"), None),
        ("inputs are not modified", no_mutation, True, "copy each record before changing it"),
        ("unknown strategy", lambda: f(train, test, "x", "average"), Raises(ValueError), None),
        ("column with no values in train", lambda: f([{"x": None}], [], "x", "mean"), Raises(ValueError), None),
    ])


def check_read_jsonl(f):
    def basic():
        path = _write(_tmp() / "log.jsonl", '{"epoch": 0, "loss": 0.9}\n\n{"epoch": 1, "loss": 0.5}\n')
        return list(f(path))

    def bad_line():
        path = _write(_tmp() / "bad.jsonl", '{"a": 1}\n{"a": 2}\n{"a": 3,,}\n')
        try:
            list(f(path))
        except ValueError as e:
            return "line 3" in str(e)
        return "no error raised"

    _scenarios("read_jsonl", [
        ("is a generator", lambda: inspect.isgenerator(f(_write(_tmp() / "g.jsonl", "{}\n"))), True, "use yield"),
        ("reads records and skips blank lines", basic, [{"epoch": 0, "loss": 0.9}, {"epoch": 1, "loss": 0.5}], None),
        ("bad JSON raises ValueError mentioning the line number", bad_line, True,
         "catch json.JSONDecodeError and raise ValueError(f'line {n}: ...')"),
    ])


def check_validate_records(f):
    schema = {
        "species": {"type": "str", "allowed": ["Adelie", "Chinstrap", "Gentoo"]},
        "body_mass_g": {"type": "float", "min": 1000, "max": 10000},
        "year": {"type": "int"},
        "sex": {"type": "str", "allowed": ["male", "female"], "required": False},
    }
    good = {"species": "Adelie", "body_mass_g": 3750.0, "year": 2007, "sex": "male"}
    records = [
        good,
        {"species": "Emperor", "body_mass_g": 3750, "year": 2007, "sex": None},
        {"species": "Gentoo", "body_mass_g": 50.0, "year": "2008"},
        {"species": "Gentoo", "body_mass_g": None, "year": 2008, "sex": "unknown"},
        {"species": "Gentoo", "body_mass_g": True, "year": 2009.0},
    ]
    _scenarios("validate_records", [
        ("a valid record passes", lambda: f([good], schema), ([good], []), None),
        ("every problem is reported", lambda: f(records, schema)[1], [
            (1, "species", "not_allowed"),
            (2, "body_mass_g", "below_min"),
            (2, "year", "type"),
            (3, "body_mass_g", "missing"),
            (3, "sex", "not_allowed"),
            (4, "body_mass_g", "type"),
            (4, "year", "type"),
        ], "check columns in schema order. An int is fine where a float is expected, but a bool never counts as a number"),
        ("only records with no errors are kept", lambda: f(records, schema)[0], [good], None),
        ("optional columns may be missing", lambda: f([{"species": "Gentoo", "body_mass_g": 5000, "year": 2008}], schema)[1], [],
         "sex has required=False, so None or an absent key is fine"),
    ])


def check_deduplicate(f):
    records = [
        {"id": "P1", "species": "Adelie", "mass": 3700},
        {"id": "P2", "species": "Gentoo", "mass": 5000},
        {"id": " p1 ", "species": "ADELIE", "mass": 3710},
        {"id": "P3", "species": "Adelie", "mass": 3700},
    ]

    def no_mutation():
        copies = [dict(r) for r in records]
        f(copies, ["id"])
        return copies == records

    _scenarios("deduplicate", [
        ("by id, ignoring case and spaces", lambda: f(records, ["id"]), ([records[0], records[1], records[3]], 1),
         "compare stripped, lowercased strings, but keep the original records unchanged in the output"),
        ("by several columns", lambda: f(records, ["species", "mass"]), ([records[0], records[1], records[2]], 1), None),
        ("all columns when keys is None", lambda: f(records, None), (records, 0),
         "with keys=None, compare every column. No two records match on all of them"),
        ("keeps the first occurrence", lambda: f([{"a": 1, "n": "x"}, {"a": 1, "n": "y"}], ["a"])[0], [{"a": 1, "n": "x"}], None),
        ("inputs are not modified", no_mutation, True, None),
    ])


def check_Checkpointer(C):
    def rotation():
        folder = _tmp() / "ckpt"
        c = C(folder, keep_last=3)
        for step in [1, 2, 3, 4, 5]:
            c.save(step, {"w": step * 0.1})
        return (c.steps(), sorted(p.name for p in folder.iterdir()))

    def latest():
        c = C(_tmp() / "ckpt", keep_last=2)
        c.save(10, {"w": 1.0})
        c.save(20, {"w": 2.0})
        return c.latest()

    def persists():
        folder = _tmp() / "ckpt"
        C(folder).save(7, {"lr": 0.01})
        return C(folder).latest()

    def ignores_other_files():
        folder = _tmp() / "ckpt"
        _write(folder / "notes.txt", "keep me")
        c = C(folder, keep_last=1)
        c.save(1, {})
        c.save(2, {})
        return sorted(p.name for p in folder.iterdir())

    def overwrite():
        c = C(_tmp() / "ckpt")
        c.save(3, {"v": 1})
        c.save(3, {"v": 2})
        return (c.steps(), c.load(3))

    _scenarios("Checkpointer", [
        ("keeps only the last N checkpoints", rotation,
         ([3, 4, 5], ["ckpt_000003.json", "ckpt_000004.json", "ckpt_000005.json"]),
         "file names are ckpt_{step:06d}.json. Delete the oldest ones after saving"),
        ("latest() returns (step, state)", latest, (20, {"w": 2.0}), None),
        ("latest() with no checkpoints", lambda: C(_tmp() / "empty").latest(), None, None),
        ("a new Checkpointer finds existing files", persists, (7, {"lr": 0.01}), "read the folder in steps(), don't only remember what this object saved"),
        ("other files in the folder are left alone", ignores_other_files, ["ckpt_000002.json", "notes.txt"], None),
        ("saving the same step again overwrites it", overwrite, ([3], {"v": 2}), None),
        ("keep_last=0 is invalid", lambda: C(_tmp(), keep_last=0), Raises(ValueError), None),
    ])


print("Setup done. Checks are ready.")

## Easy

### Exercise 1: `parse_value`

Write `parse_value(text)` that converts one raw field from a CSV file into a proper Python value:

- Strip spaces first.
- These mean "missing" and become `None` (in any capitalisation): an empty string, `NA`, `N/A`, `nan`, `null`, `none`, `?`, `-`
- `true` and `false` (in any capitalisation) become `True` and `False`.
- Whole numbers become `int`, like `"42"` to `42`.
- Other numbers become `float`, like `"3.5"` or `"1e-3"`.
- Anything else stays a string (stripped).

```python
parse_value(" NA ")    # None
parse_value("42")      # 42
parse_value("3.5")     # 3.5
parse_value("TRUE")    # True
parse_value(" Adelie ")   # "Adelie"
```

You'll reuse this in exercise 5.

In [ ]:
MISSING_MARKERS = {"", "na", "n/a", "nan", "null", "none", "?", "-"}


def parse_value(text: str):
    # your code here
    pass

In [ ]:
check_parse_value(parse_value)

### Exercise 2: `count_lines`

Write `count_lines(path)` that returns the number of lines in a text file.

- The last line counts even if it doesn't end with a newline.
- Blank lines count.
- An empty file has 0 lines.
- `path` may be a string or a `Path`.
- Read the file lazily, so it works on files much bigger than memory.

In [ ]:
def count_lines(path) -> int:
    # your code here
    pass

In [ ]:
check_count_lines(count_lines)

### Exercise 3: `save_json`

Write `save_json(obj, path)` that saves an object as a JSON file:

- Create any missing parent folders.
- Use 2-space indentation, and keep the keys in their original order.
- Write non-English characters as-is (`"Adélie"`, not `"Ad\u00e9lie"`), using UTF-8.
- End the file with a newline.
- Return the path as a `Path`.

```python
save_json({"b": 1, "a": [1, 2]}, "out/result.json")
```

should create `out/result.json` containing exactly:

```
{
  "b": 1,
  "a": [
    1,
    2
  ]
}
```

In [ ]:
import json
from pathlib import Path


def save_json(obj, path) -> Path:
    # your code here
    pass

In [ ]:
check_save_json(save_json)

### Exercise 4: `find_files`

Write `find_files(folder, suffix)` that finds every **file** with the given extension in `folder` and all its subfolders.

- Match the suffix without caring about capitals: `".csv"` matches `data.CSV`.
- Skip folders, even if their name ends with the suffix.
- Return a sorted list of paths **relative to `folder`**, as strings with forward slashes, like `"sub/c.csv"`.
- `folder` may be a string or a `Path`.

```python
find_files("dataset", ".jpg")
# ["train/cat_001.jpg", "train/cat_002.jpg", "val/cat_101.jpg", ...]
```

Image datasets are usually organised this way: one folder per split or per class, with the files inside.

In [ ]:
from pathlib import Path


def find_files(folder, suffix: str) -> list[str]:
    # your code here
    pass

In [ ]:
check_find_files(find_files)

## Medium

### Exercise 5: `read_csv_records`

Write `read_csv_records(path)` that reads a CSV file into a list of dicts, one per row, with every value converted by your `parse_value` from exercise 1.

- Use the `csv` module so quoted values with commas work.
- Files saved by Excel often start with an invisible byte order mark (BOM). Make sure the first column name comes out correctly anyway.

```python
# people.csv:
# name,age,city
# "Smith, John",42,Kathmandu
# Ana,NA,Pokhara

read_csv_records("people.csv")
# [{"name": "Smith, John", "age": 42, "city": "Kathmandu"},
#  {"name": "Ana", "age": None, "city": "Pokhara"}]
```

In [ ]:
import csv


def read_csv_records(path) -> list[dict]:
    # your code here
    pass

In [ ]:
check_read_csv_records(read_csv_records)

### Exercise 6: `write_csv`

Write `write_csv(records, path)` that writes a list of dicts to a CSV file and returns the number of rows written.

- The header contains every key that appears in **any** record, in the order they're first seen.
- A missing key or a `None` value becomes an empty field. Other values, including `0` and `False`, are written normally.
- Use the `csv` module, so values containing commas are quoted correctly.

```python
write_csv([{"name": "Smith, John", "age": 42}, {"name": "Ana", "city": "Pokhara"}], "out.csv")
```

writes:

```
name,age,city
"Smith, John",42,
Ana,,Pokhara
```

In [ ]:
import csv


def write_csv(records: list[dict], path) -> int:
    # your code here
    pass

In [ ]:
check_write_csv(write_csv)

### Exercise 7: `missing_report`

Write `missing_report(records)` that summarises missing values in a list of dicts.

- Consider every column that appears in any record.
- A value counts as missing if it's `None` **or** the key is absent from that record.
- Return a list of `(column, n_missing, fraction_missing)` tuples, sorted by `n_missing` from high to low, then by column name.
- An empty list of records gives an empty report.

```python
records = [
    {"species": "Adelie", "mass": 3700, "sex": None},
    {"species": "Gentoo", "mass": None},
]
missing_report(records)
# [("sex", 2, 1.0), ("mass", 1, 0.5), ("species", 0, 0.0)]
```

In [ ]:
def missing_report(records: list[dict]) -> list[tuple]:
    # your code here
    pass

In [ ]:
check_missing_report(missing_report)

### Exercise 8: `fill_missing`

Write `fill_missing(train, test, column, strategy="median")` that fills in missing (`None`) values in one column of two lists of records.

- `strategy` is `"median"`, `"mean"` or `"mode"` (most common value; ties go to the value seen first). Anything else raises a `ValueError`.
- **The fill value is computed from the training records only**, ignoring their missing values. Then it's used to fill both lists.
- Raise a `ValueError` if the training data has no values at all in that column.
- Don't modify the inputs. Return `(new_train, new_test, fill_value)`.

```python
train = [{"x": 1.0}, {"x": None}, {"x": 3.0}, {"x": 10.0}]
test = [{"x": None}, {"x": 5.0}]
fill_missing(train, test, "x", "median")
# ([{"x": 1.0}, {"x": 3.0}, {"x": 3.0}, {"x": 10.0}], [{"x": 3.0}, {"x": 5.0}], 3.0)
```

In [ ]:
def fill_missing(train: list[dict], test: list[dict], column: str, strategy: str = "median"):
    # your code here
    pass

In [ ]:
check_fill_missing(fill_missing)

### Exercise 9: `read_jsonl`

Write a **generator** `read_jsonl(path)` that reads a JSON Lines file and yields one dict per line.

- Skip blank lines.
- If a line isn't valid JSON, raise a `ValueError` whose message includes the line number, like `"line 3: ..."`. Line numbers start at 1 and count blank lines too.

```python
for record in read_jsonl("training_log.jsonl"):
    print(record["epoch"], record["loss"])
```

In [ ]:
import json


def read_jsonl(path):
    # your code here
    pass

In [ ]:
check_read_jsonl(read_jsonl)

## Hard

### Exercise 10: `validate_records`

Write `validate_records(records, schema)` that checks records against a schema and reports every problem.

The schema maps column names to rules:

```python
schema = {
    "species": {"type": "str", "allowed": ["Adelie", "Chinstrap", "Gentoo"]},
    "body_mass_g": {"type": "float", "min": 1000, "max": 10000},
    "year": {"type": "int"},
    "sex": {"type": "str", "allowed": ["male", "female"], "required": False},
}
```

For each record, check each column **in schema order**:

- If the value is `None` or the key is absent: that's a `"missing"` error, unless the rule has `"required": False`, in which case it's fine. Either way, skip the other checks for that column.
- `"type"`: `"str"` needs a `str`. `"int"` needs an `int`. `"float"` accepts an `int` or a `float`. A `bool` never counts as a number. Wrong type gives a `"type"` error, and the remaining checks for that column are skipped.
- `"allowed"`: the value must be in the list, otherwise `"not_allowed"`.
- `"min"` / `"max"`: otherwise `"below_min"` / `"above_max"`.

Return `(valid_records, errors)`, where `valid_records` are the records with no errors at all, and `errors` is a list of `(record_index, column, error_code)` tuples.

In [ ]:
def validate_records(records: list[dict], schema: dict):
    # your code here
    pass

In [ ]:
check_validate_records(validate_records)

### Exercise 11: `deduplicate`

Write `deduplicate(records, keys)` that removes duplicate records, keeping the first occurrence of each.

- Two records are duplicates if they match on every column in `keys`. If `keys` is `None`, compare on every column that appears in any record (an absent column counts as `None`).
- When comparing, strings are stripped and lowercased, so `" p1 "` matches `"P1"`. Other values are compared as they are.
- The records you return are the **original, unmodified** records.
- Return `(unique_records, number_removed)`.

```python
records = [
    {"id": "P1", "mass": 3700},
    {"id": "P2", "mass": 5000},
    {"id": " p1 ", "mass": 3710},
]
deduplicate(records, ["id"])
# ([{"id": "P1", "mass": 3700}, {"id": "P2", "mass": 5000}], 1)
```

In [ ]:
def deduplicate(records: list[dict], keys: list[str] | None = None):
    # your code here
    pass

In [ ]:
check_deduplicate(deduplicate)

### Exercise 12: `Checkpointer`

Long training runs save **checkpoints** so they can resume after a crash. Since checkpoints can be large, old ones are deleted automatically. Write a `Checkpointer(folder, keep_last=3)` class:

- The folder is created if it doesn't exist. `keep_last` below 1 raises a `ValueError`.
- `save(step, state)` writes `{"step": step, "state": state}` as JSON to `folder/ckpt_{step:06d}.json` (so step 42 is `ckpt_000042.json`), then deletes the oldest checkpoint files so only `keep_last` remain. Saving an existing step overwrites it.
- `steps()` returns the sorted list of steps that have a checkpoint file **in the folder**, so a new `Checkpointer` on the same folder sees checkpoints saved earlier.
- `load(step)` returns the saved `state` for that step.
- `latest()` returns `(step, state)` for the highest step, or `None` if there are none.
- Leave any other files in the folder alone.

```python
ckpt = Checkpointer("checkpoints", keep_last=3)
for step in range(1, 6):
    ckpt.save(step, {"weights": [...], "lr": 0.01})
ckpt.steps()    # [3, 4, 5]
```

In [ ]:
import json
from pathlib import Path


class Checkpointer:
    # your code here
    pass

In [ ]:
check_Checkpointer(Checkpointer)

## Done?

Once everything passes, compare with `03_solutions.ipynb`. Then head to the mini project in `04_mini_project.ipynb`, where you'll take the raw penguin CSV all the way to a trained classifier saved as a JSON file.